# Fine-Tuning a Small LLM with LoRA for OCR Error Correction

**Goal:** Fine-tune `Qwen2.5-0.5B-Instruct` using LoRA (PEFT) to correct common OCR errors in document text — e.g. `1D card` → `ID card`, `rnodel number` → `model number`, `O123456` → `0123456`.

This mirrors a real document-intelligence problem: OCR pipelines (used for IDs, invoices, forms) often misread visually similar characters. A lightweight LLM fine-tuned specifically for this correction task can sit downstream of an OCR engine to clean up its output before the text is used for identity/document verification.

**Stack:** PyTorch, Hugging Face `transformers`, `peft` (LoRA), `trl` (SFTTrainer), `bitsandbytes` (4-bit quantization) — runs on a free Colab T4 GPU.

**Pipeline:**
1. Build a synthetic OCR-noise dataset (programmatic character-level corruption of clean document-style sentences)
2. Load a small base model in 4-bit
3. Attach a LoRA adapter
4. Fine-tune with SFTTrainer
5. Evaluate qualitatively (before/after correction examples)
6. Save the adapter for reuse / upload to Hugging Face Hub

## 1. Setup

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. Build a synthetic OCR-error dataset

We start from clean, document-style sentences (ID cards, invoices, forms, addresses) and programmatically inject realistic OCR confusions:
- `l` ↔ `1` ↔ `I`
- `O` ↔ `0`
- `rn` ↔ `m`
- `S` ↔ `5`, `B` ↔ `8`, `Z` ↔ `2`
- Random dropped/doubled characters (common at low OCR confidence)

This keeps the project fully self-contained and reproducible (no external dataset download needed), while still reflecting the actual error distribution OCR engines produce.

In [ ]:
import random

random.seed(42)

CLEAN_SENTENCES = [
    "Name: Rajesh Kumar Sharma",
    "Date of Birth: 15/08/1995",
    "Permanent Account Number: ABCDE1234F",
    "Address: 221B, MG Road, Bangalore, Karnataka 560001",
    "Invoice Number: INV-2026-004521",
    "Total Amount Due: Rs. 45,600.00",
    "Aadhaar Number: 1234 5678 9012",
    "Passport Number: M1234567",
    "Bank Account Number: 0123456789012345",
    "IFSC Code: HDFC0001234",
    "Vehicle Registration Number: KA01AB1234",
    "Policy Number: LIC998877665544",
    "GST Identification Number: 29ABCDE1234F1Z5",
    "Employee ID: EMP0045123",
    "Date of Issue: 01/01/2024, Valid Until: 31/12/2029",
    "Contact Number: +91 8409388760",
    "Order Reference: ORD/2026/778899",
    "Pin Code: 560034",
    "Blood Group: O Positive",
    "Signature verified by branch manager on 12/03/2026",
]

CONFUSIONS = [
    ("l", "1"), ("I", "1"), ("O", "0"), ("o", "0"),
    ("S", "5"), ("B", "8"), ("Z", "2"), ("m", "rn"),
    ("G", "6"), ("D", "0"),
]

def corrupt(text, n_edits=3):
    chars = list(text)
    for _ in range(n_edits):
        idx = random.randint(0, len(chars) - 1)
        op = random.choice(["swap", "drop", "double"])
        if op == "swap":
            for a, b in CONFUSIONS:
                if chars[idx] == a:
                    chars[idx] = b
                    break
                elif chars[idx] == b:
                    chars[idx] = a
                    break
        elif op == "drop" and len(chars) > 5:
            del chars[idx]
        elif op == "double":
            chars.insert(idx, chars[idx])
    return "".join(chars)

def build_dataset(n_per_sentence=25):
    examples = []
    for sent in CLEAN_SENTENCES:
        for _ in range(n_per_sentence):
            noisy = corrupt(sent, n_edits=random.randint(1, 4))
            examples.append({"noisy": noisy, "clean": sent})
    random.shuffle(examples)
    return examples

raw_data = build_dataset()
print(f"Total examples: {len(raw_data)}")
for ex in raw_data[:5]:
    print("NOISY:", ex["noisy"])
    print("CLEAN:", ex["clean"])
    print()

In [ ]:
from datasets import Dataset

PROMPT_TEMPLATE = (
    "<|im_start|>system\nYou are an OCR post-processing assistant. "
    "Correct OCR errors in the given document text and return only the corrected text."
    "<|im_end|>\n"
    "<|im_start|>user\nFix the OCR errors in this text:\n{noisy}<|im_end|>\n"
    "<|im_start|>assistant\n{clean}<|im_end|>"
)

def format_example(ex):
    return {"text": PROMPT_TEMPLATE.format(noisy=ex["noisy"], clean=ex["clean"])}

formatted = [format_example(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted)

split = dataset.train_test_split(test_size=0.1, seed=42)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds)
print(eval_ds)

## 3. Load base model in 4-bit + attach LoRA adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # small enough for a free T4

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

## 4. Train with SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./ocr-lora-checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_seq_length=256,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

trainer.train()

## 5. Qualitative evaluation: before vs. after fine-tuning

In [ ]:
def correct_text(noisy_text, model, tokenizer, max_new_tokens=64):
    prompt = (
        "<|im_start|>system\nYou are an OCR post-processing assistant. "
        "Correct OCR errors in the given document text and return only the corrected text."
        "<|im_end|>\n"
        f"<|im_start|>user\nFix the OCR errors in this text:\n{noisy_text}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.strip()

test_examples = [eval_ds[i] for i in range(5)]
for ex in test_examples:
    noisy = ex["text"].split("Fix the OCR errors in this text:\n")[1].split("<|im_end|>")[0]
    prediction = correct_text(noisy, model, tokenizer)
    print("NOISY:     ", noisy)
    print("PREDICTED: ", prediction)
    print("-" * 60)

## 6. Save the LoRA adapter

This saves only the small adapter weights (a few MB), not the full base model — the standard, efficient way to share a fine-tuned result.

In [ ]:
ADAPTER_DIR = "./ocr-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"Adapter saved to {ADAPTER_DIR}")

### Optional: push to the Hugging Face Hub

```python
from huggingface_hub import login
login()  # paste your HF token when prompted

model.push_to_hub("your-username/qwen2.5-0.5b-ocr-lora")
tokenizer.push_to_hub("your-username/qwen2.5-0.5b-ocr-lora")
```

### Next steps if you want to extend this project
- Swap the synthetic dataset for real OCR output (run Tesseract/EasyOCR on scanned ID/invoice images and pair it with ground-truth text)
- Compare LoRA rank (r=8/16/32) and report accuracy/edit-distance tradeoffs
- Add a held-out benchmark with character error rate (CER) as a quantitative metric
- Try a small VLM (e.g. a compact LLaVA-style model) to correct errors directly from the document image instead of pre-OCR'd text